[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/python_binding_examples/lsdtt3_bindings_quickstart.ipynb)

# Channels, basins, slope–area and chi in Calabria with the lsdtt3 Python bindings

The other notebooks in this repository run the `lsdtt3` **command-line programs**, controlled by parameter files. This one does the analysis with the `lsdtt3` **Python bindings** instead: every step is a Python function call, and the results come back as Python objects (rasters, tables and GeoPandas layers) rather than files. We use `lsdviztools3` to draw the maps and charts.

Our example is the Aspromonte and the Serre mountains of Calabria, southern Italy, the same area as the *landscape transience* notebook, so you can compare the two. In this notebook you will:

1. Install `lsdviztools3` and the `lsdtt3` Python wheel.
2. Download a 30 m DEM with `lsdtt3.fetch_dem`, and remove the sea.
3. Make a hillshade and a map of topographic gradient.
4. Route flow and extract the channel network.
5. Extract six drainage basins from their outlet points.
6. Make a **slope–area plot** of the basins.
7. Run a **chi analysis** of the basins, and plot chi profiles and channel steepness.

**The wheel is for Linux x86_64 and Python 3.13 only**, which is what Google Colab provides. The notebook checks this before installing it.

## READ THIS FIRST

**To use your own landscape you only need to change one cell**, the one under **"Edit code block below for your site!"**: the two corners of your box, and the basin outlets. You get all of them by right-clicking on Google Maps.

Every figure is saved as a png, with a file name that tells you what it is.

## Setup

### First set up condacolab

`lsdviztools3` draws its maps with GMT (the Generic Mapping Tools), which we install with conda. This step takes around 2 minutes. **`condacolab.install()` restarts the Colab runtime.** You will see a message that the session crashed: this is expected. Wait for it to reconnect, then carry on with the next cell.

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

Now install `pygmt`, the Python interface to GMT. This is the slowest step and takes around a minute.

In [ ]:
# Install pygmt. mamba prints hundreds of lines, so we hide them and only
# show the end of the output if something goes wrong.
import subprocess

result = subprocess.run(["mamba", "install", "-y", "pygmt"], capture_output=True, text=True)
if result.returncode == 0:
    print("pygmt installed successfully.")
else:
    print("pygmt installation FAILED. The last lines of the output were:\n")
    print("\n".join((result.stdout + result.stderr).splitlines()[-30:]))

### Get lsdviztools3

We install it with its `charts` extra, which adds `matplotlib` for the slope–area and chi plots. (Keep the quotes: they stop the shell from misreading the square brackets.)

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdviztools3-0.2.0-py3-none-any.whl

In [ ]:
!pip install "lsdviztools3-0.2.0-py3-none-any.whl[charts]"

### Get the lsdtt3 Python wheel

The wheel contains compiled code for one version of Python, so check the version first. It must be **3.13**.

In [ ]:
import sys

print("Python", sys.version.split()[0])
if sys.version_info[:2] != (3, 13):
    raise RuntimeError(
        f"This lsdtt3 wheel is for Python 3.13, but this session runs Python "
        f"{sys.version_info.major}.{sys.version_info.minor}. Ask for a wheel for this version.")

In [ ]:
!pip install https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdtt3-0.7.1-cp313-cp313-manylinux_2_28_x86_64.whl

Now import `lsdtt3`.

`lsdtt3` carries its own copy of the PROJ and GDAL data files (the coordinate system database, among other things) and points itself at them internally. The 0.7.1 wheel also sets the environment variables `PROJ_DATA`, `PROJ_LIB` and `GDAL_DATA` when it is imported (later versions do not). Those would make the *other* geospatial libraries in this session (pyproj, and GMT behind `lsdviztools3`) read `lsdtt3`'s copy of the PROJ database, which may be older than the one they expect, and then fail. `lsdtt3` does not need the variables, so we remove them straight after the import.

In [ ]:
import os

import lsdtt3

for name in ("PROJ_DATA", "PROJ_LIB", "GDAL_DATA"):
    os.environ.pop(name, None)

print("lsdtt3", lsdtt3.__version__, "with GDAL", lsdtt3._core.gdal_version())

## Your site

### Reading latitude and longitude from Google Maps

This cell defines two small functions that read latitude and longitude pairs copied from Google Maps. You do not need to edit it; just run it.

In [ ]:
import math
import re


def parse_lat_lon(text):
    """Read a "latitude, longitude" pair, as copied from Google Maps, into two numbers."""
    numbers = re.findall(r"[-+]?\d+(?:\.\d+)?", text)
    if len(numbers) != 2:
        raise ValueError(
            f'Could not find a latitude and a longitude in "{text}". Right-click the place '
            'on Google Maps, click the two numbers at the top of the menu to copy them, and '
            'paste them between the quotes, for example "55.9476, -3.1582".')
    latitude = float(numbers[0])
    longitude = float(numbers[1])
    if not -90 <= latitude <= 90:
        raise ValueError(f'"{text}": the first number is the latitude and must be between '
                         '-90 and 90. Google Maps gives the latitude first.')
    if not -180 <= longitude <= 180:
        raise ValueError(f'"{text}": the second number is the longitude and must be '
                         'between -180 and 180.')
    return latitude, longitude


def box_from_corners(lower_left, upper_right):
    """Return west, east, south and north for the box with these two opposite corners."""
    lat_1, lon_1 = parse_lat_lon(lower_left)
    lat_2, lon_2 = parse_lat_lon(upper_right)
    if lat_1 == lat_2 or lon_1 == lon_2:
        raise ValueError("The two corners must have different latitudes and different longitudes.")
    # min and max, so you get the same box whichever pair of opposite corners you pick
    west, east = min(lon_1, lon_2), max(lon_1, lon_2)
    south, north = min(lat_1, lat_2), max(lat_1, lat_2)
    width_km = (east - west) * 111.32 * math.cos(math.radians((south + north) / 2))
    height_km = (north - south) * 110.57
    print(f"Box: west {west:.5f}, east {east:.5f}, south {south:.5f}, north {north:.5f}")
    print(f"Size: about {width_km:.0f} km east-west by {height_km:.0f} km north-south")
    return west, east, south, north

### Edit code block below for your site!

**The box.** Choose it with two right-clicks on Google Maps:

1. On Google Maps, right-click the **lower left (south-west)** corner of your area. The first line of the menu that appears is the latitude and longitude, for example `55.9476273103734, -3.158220317286066`. Click it to copy it.
2. Paste it between the quotes after `lower_left =` (Ctrl-V, or Cmd-V on a Mac).
3. Do the same for the **upper right (north-east)** corner, after `upper_right =`.

If you pick the other two corners instead (upper left and lower right), you get the same box.

**The basin outlets.** Right-click a point on each river, at the downstream end of the basin you want, copy it, and paste it as one line of `outlets`, in quotes and followed by a comma. Each basin must lie **completely** inside the box, from its outlet all the way up to its drainage divide.

**Other settings:**

* `prefix`: every file the notebook writes starts with this.
* `threshold_pixels`: a channel starts where this many pixels drain to a point. With 30 m pixels that is 500 × 900 m² = 0.45 km².
* `m_over_n`: the concavity index used for chi (see Step 6).

In [ ]:
prefix = "calabria"

# Right-click the lower left (south-west) corner on Google Maps, copy, paste:
lower_left = "37.87521641344407, 15.628317625157045"
# ... and the upper right (north-east) corner:
upper_right = "38.30554438686832, 16.210748067627133"

# One "latitude, longitude" per basin outlet, each on or just upstream of a river
outlets = [
    "37.94234239772101, 15.790076672680563",
    "38.093114, 15.680576",
    "38.06143176957989, 16.060604039819015",
    "38.099083, 15.677132",
    "38.20455853467961, 15.67642588617276",
    "38.12546843046537, 16.1230887795859",
]

threshold_pixels = 500
m_over_n = 0.45

west, east, south, north = box_from_corners(lower_left, upper_right)
outlet_lat_lon = [parse_lat_lon(text) for text in outlets]
print(f"{len(outlet_lat_lon)} outlets")

## Step 1: Download a DEM

`lsdtt3.fetch_dem` downloads the DEM tiles that cover the box, joins them and returns an in-memory `Raster`:

* `source="cop30_aws"`: the Copernicus GLO-30 DEM (about 30 m resolution), served from AWS. No account or API key is needed.
* `grid_spacing=30.0`: a 30 m output grid.
* We leave `target_epsg` at its default, `0`, which means "the UTM zone at the centre of the box" (here zone 33N, EPSG:32633). UTM is a coordinate system in metres: gradient, flow routing and chi all need distances in metres, not degrees, and square pixels.

**Important**: you should always cite your data source. For this data, you should cite:

*European Space Agency, Sinergise (2021). Copernicus Global Digital Elevation Model. Distributed by OpenTopography. https://doi.org/10.5069/G9028PQB*

In [ ]:
dem_raw = lsdtt3.fetch_dem(west=west, south=south, east=east, north=north,
                           source="cop30_aws", grid_spacing=30.0)

meta = dem_raw.metadata
dem_crs = meta.crs_wkt          # the DEM's coordinate system, as text
n_rows, n_cols = dem_raw.shape
print(f"DEM: {n_rows} rows by {n_cols} columns of {meta.cell_size:.0f} m pixels")
print("UTM EPSG code for the box centre:",
      lsdtt3.utm_epsg_for_point((west + east) / 2, (south + north) / 2))

### Remove the sea

The Copernicus DEM does not mark the sea as nodata: it stores it as an elevation of exactly **0 m**. Left in, the sea is a huge, perfectly flat surface, which spoils the gradient map, the slope–area plot and the flow routing. `lsdtt3.mask_to_nodata(dem, 0.0)` sets every pixel at or below 0 m to nodata.

We also write the DEM to a GeoTIFF file, because `lsdviztools3` draws its maps from files.

In [ ]:
dem = lsdtt3.mask_to_nodata(dem_raw, 0.0)
print(f"Removed {dem_raw.valid_pixel_count - dem.valid_pixel_count:,} sea pixels "
      f"({dem.valid_pixel_count:,} land pixels left)")
dem.write(f"{prefix}_DEM.tif")

## Step 2: Hillshade and topographic gradient

* `lsdtt3.hillshade` lights the topography from the northwest (azimuth 315°), 45° above the horizon, which makes a good background map.
* `lsdtt3.polyfit_metrics` fits a smooth polynomial surface to the elevations within `lambda_metres` (here 70 m, which suits 30 m data) of each pixel, and takes the gradient of that surface. `"slope"` is the **topographic gradient in m/m** (rise over run). The method follows:

*Hurst, M.D., Mudd, S.M., Walcott, R., Attal, M., Yoo, K., 2012. Using hilltop curvature to derive the spatial distribution of erosion rates. J. Geophys. Res. 117, F02017. https://doi.org/10.1029/2011JF002057*

In [ ]:
hillshade = lsdtt3.hillshade(dem, azimuth=315.0, altitude=45.0)
hillshade.write(f"{prefix}_hillshade.tif")

slope = lsdtt3.polyfit_metrics(dem, ["slope"], lambda_metres=70.0)["slope"]
slope.write(f"{prefix}_slope.tif")

import numpy as np

slope_values = slope.to_numpy()[slope.mask_numpy()]
print(f"Median gradient: {float(np.median(slope_values)):.3f} m/m")

### Figure 1: Shaded relief

From here on we draw with `lsdviztools3`. `render_hillshade` draws the hillshade; `shade=False` because the raster is already a hillshade, and we use a grey colour map. The sea is nodata, so it is left blank.

In [ ]:
from dataclasses import replace

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from lsdviztools3.render.style import MapStyle
from lsdviztools3.render.hillshade import render_hillshade
from lsdviztools3.render.channels import render_drape, render_channels
from lsdviztools3.render.basins import render_basins
from lsdviztools3.render.points import render_points
from lsdviztools3.render.base import save_figure

style = MapStyle(figure_size="15c", transparent=False)
hs_style = replace(style, cmap="gray")

fig = render_hillshade(f"{prefix}_hillshade.tif", shade=False, style=hs_style)
save_figure(fig, f"{prefix}_relief_plot.png", style=style)
display(Image(f"{prefix}_relief_plot.png", width=800))

### Figure 2: Topographic gradient

`render_drape` draws the hillshade in grey and lays the gradient on top in colour. `transparency` sets how much of the hillshade shows through (0 = solid colour, 100 = invisible). We clip the colour scale at the 2nd and 98th percentiles so a few extreme pixels do not wash out the map.

In [ ]:
fig = render_drape(f"{prefix}_hillshade.tif", f"{prefix}_slope.tif",
                   base_is_hillshade=True, transparency=30, cmap="inferno",
                   percentile_min=2, percentile_max=98,
                   colorbar=True, colorbar_label="Gradient (m/m)", colorbar_position="right",
                   style=style)
save_figure(fig, f"{prefix}_gradient_plot.png", style=style)
display(Image(f"{prefix}_gradient_plot.png", width=800))

## Step 3: Flow routing and the channel network

* `lsdtt3.fill_pits` fills the pits in the DEM, so that water can flow off the landscape everywhere.
* `lsdtt3.FlowInfo` routes flow with the D8 method: each pixel drains to its steepest downhill neighbour. It holds the flow directions, drainage areas and much more, and every later step uses it.
* `flow.channel_scope(threshold_pixels)` finds the channel heads (sources): a channel starts wherever `threshold_pixels` pixels drain to a point.
* `flow.channel_network(scope)` builds the network downstream from those heads, split into links between junctions.
* `network.lines_geodataframe()` returns the links as a GeoPandas layer, with a `stream_order` (Strahler order) column.

In [ ]:
filled = lsdtt3.fill_pits(dem)
flow = lsdtt3.FlowInfo(filled)

network = flow.channel_network(flow.channel_scope(threshold_pixels))
channel_lines = network.lines_geodataframe()

print(f"{network.n_channel_cells:,} channel pixels, {network.n_junctions:,} junctions")
print("Number of channel links in each stream order:")
print(channel_lines["stream_order"].value_counts().sort_index())

### Figure 3: The channel network

We draw each stream order separately: higher-order (bigger) rivers get thicker, darker lines. In Strahler ordering, two order-1 streams join to make an order-2 stream, two order-2 streams make an order-3 stream, and so on.

In [ ]:
colours = {1: "lightskyblue", 2: "deepskyblue", 3: "dodgerblue",
           4: "blue", 5: "navy", 6: "midnightblue", 7: "black"}

fig = render_hillshade(f"{prefix}_hillshade.tif", shade=False, style=hs_style)
for order in sorted(channel_lines["stream_order"].unique()):
    subset = channel_lines[channel_lines["stream_order"] == order]
    order_style = replace(style,
                          line_width=f"{0.3 * order:.1f}p",
                          line_color=colours.get(int(order), "black"))
    fig = render_channels(subset, style=order_style, fig=fig)
save_figure(fig, f"{prefix}_channels_plot.png", style=style)
display(Image(f"{prefix}_channels_plot.png", width=800))

## Step 4: Extract the basins

### Snapping the outlets to the rivers

A point picked on Google Maps is rarely exactly on the river pixel that the flow routing found. So we **snap** each outlet: we convert its latitude and longitude to the DEM's coordinates, find its pixel, and then follow the flow path downstream (`flow.receiver_of`) until we reach a pixel with at least `threshold_pixels` upslope pixels, i.e. a channel pixel. This is what `basin_outlet_snapping_method: downstream` does in the command-line programs. So pick each outlet on, or just upstream of, the river.

`flow.basins(...)` then takes the outlet pixels (as rows and columns) and returns the basins. Its `table` says, for each basin, whether it is **complete**: a basin cut by the edge of the DEM, or by nodata such as the sea, is not, and its chi profile would be wrong. If a basin is incomplete, make the box bigger.

In [ ]:
import geopandas as gpd
import numpy as np
from pyproj import Transformer

to_dem = Transformer.from_crs("EPSG:4326", dem_crs, always_xy=True)
contributing = flow.contributing_pixels().to_numpy()


def snap_to_channel(latitude, longitude):
    """Row and column of the first channel pixel downstream of a point."""
    x, y = to_dem.transform(longitude, latitude)
    col = int((x - meta.x_min) // meta.cell_size)
    row = int((meta.y_max - y) // meta.cell_size)
    if not (0 <= row < n_rows and 0 <= col < n_cols):
        raise ValueError(f"The outlet at {latitude}, {longitude} is outside the DEM.")
    while contributing[row, col] < threshold_pixels:
        downstream = flow.receiver_of(row, col)
        if downstream is None or tuple(downstream) == (row, col):
            break                     # reached the edge of the DEM or the sea
        row, col = downstream
    return row, col


outlet_pixels = np.array([snap_to_channel(lat, lon) for lat, lon in outlet_lat_lon])
basins = flow.basins(outlet_pixels)

basin_table = basins.table
basin_table["area_km2"] = (basin_table["area_metres2"] / 1e6).round(1)
print(basin_table[["basin_id", "outlet_row", "outlet_col", "area_km2", "complete"]].to_string(index=False))
if not basin_table["complete"].all():
    print("\nWARNING: some basins are not complete. Make the box bigger, or move those outlets.")

# Layers for the map: basin outlines, and the outlets before and after snapping
basin_polygons = basins.polygons
basin_polygons["label"] = basin_polygons["basin_id"].astype(int)

requested_outlets = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy([lon for lat, lon in outlet_lat_lon],
                                [lat for lat, lon in outlet_lat_lon]),
    crs="EPSG:4326")
snapped_outlets = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(meta.x_min + (outlet_pixels[:, 1] + 0.5) * meta.cell_size,
                                meta.y_max - (outlet_pixels[:, 0] + 0.5) * meta.cell_size),
    crs=dem_crs)

### Figure 4: The basins

The basins are drawn semi-transparent and numbered; the points we picked are red crosses and the snapped outlets black circles.

In [ ]:
fig = render_hillshade(f"{prefix}_hillshade.tif", shade=False, style=hs_style)
fig = render_basins(basin_polygons, label_col="label", fill="orange", transparency=50,
                    style=replace(style, line_width="0.8p", line_color="black"), fig=fig)
fig = render_points(requested_outlets, symbol="x", pen="1.5p,red",
                    style=replace(style, point_size="0.3c"), fig=fig)
fig = render_points(snapped_outlets, symbol="c", pen="0.5p,black",
                    style=replace(style, point_size="0.2c"), fig=fig)
save_figure(fig, f"{prefix}_basins_plot.png", style=style)
display(Image(f"{prefix}_basins_plot.png", width=800))

## Step 5: A slope–area plot

Plot the topographic gradient of every pixel against its upslope drainage area, on log–log axes, and two kinds of behaviour appear:

* **Hillslopes** (small areas): the gradient increases downslope, away from the drainage divide.
* **Channels** (large areas): the gradient decreases as the drainage area grows. The stream power law predicts a straight line on log–log axes, $S = k_{sn} A^{-\theta}$, with a gradient of $-\theta = -m/n$.

The change between the two is usually near the channel heads, so the plot is a check on the channel threshold we chose.

We use the pixels inside our basins only: `basins.labels` is a raster of basin numbers (0 outside the basins), and `lsdtt3.mask_by_binary` sets everything outside them to nodata. `plot_slope_area` colours the channel pixels (from `network.channel_mask`) blue and the hillslope pixels grey, adds the median gradient in bins of drainage area (red) and a reference line with gradient $-m/n$. The dashed line marks our channel threshold, `threshold_pixels` × the pixel area.

`plot_slope_area` reads rasters from files, so we write them first.

In [ ]:
in_basins = lsdtt3.threshold_above(basins.labels, 0.5)

lsdtt3.mask_by_binary(slope, in_basins).write(f"{prefix}_basin_slope.tif")
lsdtt3.mask_by_binary(flow.drainage_area(), in_basins).write(f"{prefix}_basin_area.tif")
network.channel_mask.write(f"{prefix}_channel_mask.tif")
pixel_area = meta.cell_size ** 2

### Figure 5: Slope–area plot

In [ ]:
from lsdviztools3.charts import plot_slope_area

fig = plot_slope_area(f"{prefix}_basin_slope.tif", f"{prefix}_basin_area.tif",
                      channel_mask_raster=f"{prefix}_channel_mask.tif",
                      m_over_n=m_over_n,
                      mark_area=threshold_pixels * pixel_area,
                      title="Slope–area plot of the basins")
fig.savefig(f"{prefix}_slope_area.png", dpi=300, bbox_inches="tight")
plt.show()

## Step 6: Chi analysis of the basins

### What is chi?

A typical channel profile is a plot of elevation against flow distance. But it is hard to see changes in channel steepness on it, because headwater channels are steeper than downstream channels. The gradient of channels scales with drainage area, so to compare steepness you need to control for drainage area. The **stream power law** says that, in steady state, channel slope $S$ and drainage area $A$ are related by

$$S = k_{sn} A^{-\theta}, \qquad \theta = m/n .$$

* $\theta = m/n$ is the **concavity index**. We use the common default **m/n = 0.45**.
* $k_{sn}$ is the **normalised channel steepness index**. In many landscapes it is closely correlated with the erosion rate. See: *Kirby, E., Whipple, K., 2001. Quantifying differential rock-uplift rates via stream profile analysis. Geology 29, 415–418. https://doi.org/10.1130/0091-7613(2001)029<0415:QDRURV>2.0.CO;2*

Integrating drainage area as you move upstream gives a transformed distance along the channel called **chi** ($\chi$):

$$\chi = \int_{0}^{x} \left(\frac{A_0}{A(x')}\right)^{m/n} dx' .$$

If you plot elevation against $\chi$, a channel in steady state is a straight line whose **gradient is $k_{sn}$** (with $A_0 = 1$ m², the `lsdtt3` default). In a **transient** landscape the chi profile has kinks, separated by knickpoints. See: *Perron, J.T., Royden, L., 2013. An integral approach to bedrock river profile analysis. Earth Surface Processes and Landforms 38, 570–576. https://doi.org/10.1002/esp.3302*

### Run it

* `network.chi(outlets=outlet_pixels, m_over_n=m_over_n)` computes chi on the channels upstream of each of our outlets.
* `.fit(filled, critical_divergence_metres=10.0)` fits each chi profile with straight-line **segments**, split wherever the data move more than 10 m from a straight line. Each segment's gradient is its $k_{sn}$.
* `fit.nodes` has one row per channel pixel (its chi, its observed and fitted elevation, and which branch it is on); `fit.segments` one row per fitted segment, with its `ksn`; `fit.summary` one row per basin.

`lsdviztools3`'s chi plots expect the column names of the command-line programs' output, `basin_index` and `k_sn`, so we rename two columns. We also give every node its map coordinates, for the $k_{sn}$ map in Figure 8.

In [ ]:
chi = network.chi(outlets=outlet_pixels, m_over_n=m_over_n)
fit = chi.fit(filled, critical_divergence_metres=10.0)
print(f"Chi computed for {chi.n_basins} basins")

chi_nodes = fit.nodes.rename(columns={"basin_id": "basin_index"})
chi_segments = fit.segments.rename(columns={"basin_id": "basin_index", "ksn": "k_sn"})
ksn_summary = fit.summary

# Map coordinates of each node, from its flat pixel index
node_row = chi_nodes["flat"].to_numpy() // n_cols
node_col = chi_nodes["flat"].to_numpy() % n_cols
chi_nodes["x"] = meta.x_min + (node_col + 0.5) * meta.cell_size
chi_nodes["y"] = meta.y_max - (node_row + 0.5) * meta.cell_size

print(f"{len(chi_nodes):,} channel nodes in {len(chi_segments):,} segments")

### Figure 6: Stacked chi profiles

`assign_node_ksn` gives every channel pixel the $k_{sn}$ of the segment it belongs to. `plot_chi_stacked` puts all the basins side by side (each shifted along the chi axis), coloured by $\log_{10} k_{sn}$ on one shared colour scale, so their steepness can be compared. In a transient landscape you will see the colour change from steep (yellow) low down to gentle (purple) higher up the profiles.

In [ ]:
from lsdviztools3.io.chi import assign_node_ksn
from lsdviztools3.charts import plot_chi_profile, plot_chi_stacked

chi_fit = assign_node_ksn(chi_nodes, chi_segments)

fig, ax = plot_chi_stacked(chi_fit, cmap="viridis")
fig.set_size_inches(10, 5)
fig.tight_layout()
fig.savefig(f"{prefix}_chi_stacked.png", dpi=300)
plt.show()

### Figure 7: One basin

`plot_chi_profile` draws a single basin, with the fitted straight-line segments on top of the data. Change `basin` to look at the others (they are numbered from 0, in the order of `outlets`).

In [ ]:
basin = 2

fig, ax = plt.subplots(figsize=(8, 5))
plot_chi_profile(chi_fit, basin=basin, ax=ax, cmap="viridis", point_size=3)
ax.set_title(f"Basin {basin}")
fig.tight_layout()
fig.savefig(f"{prefix}_chi_fit_basin{basin}.png", dpi=300)
plt.show()

`fit.summary` gives $k_{sn}$ for each basin. The `main_stem_*` columns use only the main stem. $k_{sn}$ here has units of m$^{2\theta}$ = m$^{0.9}$.

In [ ]:
ksn_summary[["basin_id", "n_segments", "median_ksn", "chi_weighted_mean_ksn",
             "main_stem_median_ksn"]].round(1)

### Figure 8: A map of $k_{sn}$

The channel pixels of our basins coloured by the $k_{sn}$ of their fitted segment, on a log scale, with the basin outlines in black. Look for where along the rivers $k_{sn}$ changes: in a transient landscape the knickpoints often sit at similar elevations, or at similar distances from the coast, in neighbouring basins. We leave out pixels with $k_{sn} \le 0$ (flat or slightly uphill fits), which cannot be shown on a log scale.

In [ ]:
ksn_points = gpd.GeoDataFrame(chi_fit[chi_fit["k_sn"] > 0],
                              geometry=gpd.points_from_xy(chi_fit.loc[chi_fit["k_sn"] > 0, "x"],
                                                          chi_fit.loc[chi_fit["k_sn"] > 0, "y"]),
                              crs=dem_crs)

basin_outlines = basin_polygons.copy()
basin_outlines["geometry"] = basin_polygons.boundary

fig = render_hillshade(f"{prefix}_hillshade.tif", shade=False, style=hs_style)
fig = render_channels(basin_outlines, style=replace(style, line_width="0.8p", line_color="black"), fig=fig)
fig = render_points(ksn_points, color_by="k_sn", log_color=True, cmap="viridis",
                    colorbar=True, colorbar_label="log@-10@- k@-sn@-",
                    style=replace(style, point_size="0.06c"), fig=fig)
save_figure(fig, f"{prefix}_ksn_plot.png", style=style)
display(Image(f"{prefix}_ksn_plot.png", width=800))

## Things to try

* **Your own site.** Change the two corners and the outlets in the settings cell, then run everything again.
* **The channel threshold.** Change `threshold_pixels` to 200 or 2000. How do the channel network (Figure 3) and the slope–area plot (Figure 5) change? Does the dashed line move closer to where the hillslope and channel limbs meet?
* **Concavity.** Change `m_over_n` to 0.3 or 0.6 and rerun Steps 5 and 6. For which value do the tributaries line up best with the main stem in Figure 7?
* **Segment fitting.** Change `critical_divergence_metres` to 5 or 30. How does the number of segments change?
* **Look inside the objects.** Everything `lsdtt3` returns is a Python object you can explore: try `flow.drainage_area().to_numpy()`, `network.points_geodataframe()`, `basins.statistics(dem)` or `fit.segments`.

## Citing lsdtt3

You can find the citation for `lsdtt3` on its GitHub page, https://github.com/LSDtopotools/lsdtt3.